In [35]:
from pathlib import Path
import polars as pl

ARCHIVO = Path(
    "C:/GitHub/tec-data-analisis/lab01/task/Merged file.xlsx"
)

if not ARCHIVO.is_file():
    raise FileNotFoundError(f"No se encontró el archivo: {ARCHIVO}")

df = pl.read_excel(
    source=ARCHIVO,
    sheet_name="Sheet1",
    engine="calamine",
    infer_schema_length=None
)

print("Dimensiones:", df.shape)
print("Columnas:", df.columns)
print("Tipos de datos:", df.schema)

df.head(10)

Dimensiones: (1531, 18)
Columnas: ['TY', 'T1', 'AU', 'JO', 'VL', 'SP', 'PY', 'DA', 'SN', 'DO', 'UR', 'KW', 'AB', 'EP', 'IS', 'T2', 'We', 'In']
Tipos de datos: Schema({'TY': String, 'T1': String, 'AU': String, 'JO': String, 'VL': String, 'SP': String, 'PY': String, 'DA': String, 'SN': String, 'DO': String, 'UR': String, 'KW': String, 'AB': String, 'EP': String, 'IS': String, 'T2': String, 'We': String, 'In': String})


C:\Users\qjari calizaya\AppData\Local\Temp\ipykernel_4120\1976742048.py:11: FutureWarning: from_arrow(<ArrowStreamExportable>) will return a Series instead of a DataFrame in 2.0. To avoid this warning, pass the ArrowStreamExportable to either `pl.DataFrame` or `pl.Series` instead based on your desired output type.
  df = pl.read_excel(


TY,T1,AU,JO,VL,SP,PY,DA,SN,DO,UR,KW,AB,EP,IS,T2,We,In
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""JOUR""","""Personalized sentiment classif…","""Zhang, You""","""Knowledge-Based Systems""","""226""","""107135""","""2021""","""2021/08/17/""","""0950-7051""","""https://doi.org/10.1016/j.knos…","""https://www.sciencedirect.com/…","""Sentiment analysis""","""Incorporating extra attributes…",null,null,null,null,null
null,null,"""Wang, Jin""",null,null,null,null,null,null,null,null,"""Text classification""",null,null,null,null,null,null
null,null,"""Zhang, Xuejie""",null,null,null,null,null,null,null,null,"""Sentence representation""",null,null,null,null,null,null
null,null,null,null,null,null,null,null,null,null,null,"""Bilinear attention""",null,null,null,null,null,null
"""JOUR""","""Multi-source aggregated classi…","""Ma, Yu""","""Information Fusion""","""91""","""515""","""2023""","""2023/03/01/""","""1566-2535""","""https://doi.org/10.1016/j.inff…","""https://www.sciencedirect.com/…","""Stock prediction""","""Predicting stock price movemen…","""528""",null,null,null,null
null,null,"""Mao, Rui""",null,null,null,null,null,null,null,null,"""Event-driven investing""",null,null,null,null,null,null
null,null,"""Lin, Qika""",null,null,null,null,null,null,null,null,"""Multi-source aggregating""",null,null,null,null,null,null
null,null,"""Wu, Peng""",null,null,null,null,null,null,null,null,"""Sentiment analysis""",null,null,null,null,null,null
null,null,"""Cambria, Erik""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [36]:
df.null_count()

TY,T1,AU,JO,VL,SP,PY,DA,SN,DO,UR,KW,AB,EP,IS,T2,We,In
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
1269,1269,358,1269,1273,1270,1269,1269,1269,1269,1269,246,1249,1474,1469,1517,1521,1527


In [37]:
# Comprobar cuántos encabezados repetidos existen
encabezados_repetidos = df.filter(
    pl.col("TY") == "TY"
)

print("Encabezados repetidos:", encabezados_repetidos.height)
encabezados_repetidos

Encabezados repetidos: 2


TY,T1,AU,JO,VL,SP,PY,DA,SN,DO,UR,KW,AB,EP,IS,T2,We,In
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""TY""","""T1""","""AU""","""JO""","""VL""","""SP""","""PY""","""DA""","""SN""","""DO""","""UR""","""KW""","""AB""","""IS""","""EP""","""T2""",null,null
"""TY""","""T1""","""AU""","""JO""","""VL""","""SP""","""PY""","""DA""","""SN""","""DO""","""UR""","""KW""","""AB""","""IS""","""EP""","""T2""",null,null


In [38]:
df_limpio = df.filter(
    pl.col("TY").fill_null("") != "TY"
)

print("Antes:", df.shape)
print("Después:", df_limpio.shape)

Antes: (1531, 18)
Después: (1529, 18)


In [39]:
df_limpio.filter(pl.col("TY") == "TY").height

0

In [40]:
df_con_id = df_limpio.with_columns(
    (
        (pl.col("TY").fill_null("") == "JOUR")
        .cast(pl.Int64)
        .cum_sum()
    ).alias("id_articulo")
)

df_con_id.select(
    ["id_articulo", "TY", "T1", "AU", "KW"]
).head(15)

id_articulo,TY,T1,AU,KW
i64,str,str,str,str
1,"""JOUR""","""Personalized sentiment classif…","""Zhang, You""","""Sentiment analysis"""
1,null,null,"""Wang, Jin""","""Text classification"""
1,null,null,"""Zhang, Xuejie""","""Sentence representation"""
1,null,null,null,"""Bilinear attention"""
2,"""JOUR""","""Multi-source aggregated classi…","""Ma, Yu""","""Stock prediction"""
…,…,…,…,…
3,null,null,"""Liu, Junfeng""","""Sentiment analysis"""
3,null,null,"""Kang, Zhongfeng""","""Key items"""
3,null,null,"""Zhou, Zenghui""","""Interpretability"""


In [41]:
df_con_id = df_limpio.with_columns(
    (
        (pl.col("TY").fill_null("") == "JOUR")
        .cast(pl.Int64)
        .cum_sum()
    ).alias("id_articulo")
)

df_con_id.select(
    ["id_articulo", "TY", "T1", "AU", "KW"]
).head(15)

id_articulo,TY,T1,AU,KW
i64,str,str,str,str
1,"""JOUR""","""Personalized sentiment classif…","""Zhang, You""","""Sentiment analysis"""
1,null,null,"""Wang, Jin""","""Text classification"""
1,null,null,"""Zhang, Xuejie""","""Sentence representation"""
1,null,null,null,"""Bilinear attention"""
2,"""JOUR""","""Multi-source aggregated classi…","""Ma, Yu""","""Stock prediction"""
…,…,…,…,…
3,null,null,"""Liu, Junfeng""","""Sentiment analysis"""
3,null,null,"""Kang, Zhongfeng""","""Key items"""
3,null,null,"""Zhou, Zenghui""","""Interpretability"""


In [42]:
df_con_id.select(
    pl.col("id_articulo").n_unique().alias("cantidad_articulos"),
    pl.col("id_articulo").min().alias("primer_id"),
    pl.col("id_articulo").max().alias("ultimo_id")
)

cantidad_articulos,primer_id,ultimo_id
u32,i64,i64
260,1,260


In [43]:
bloques_articulos = (
    df_con_id
    .group_by("id_articulo", maintain_order=True)
    .agg(
        pl.len().alias("numero_filas"),
        pl.col("T1").first().alias("titulo")
    )
)

bloques_articulos.head(10)

id_articulo,numero_filas,titulo
i64,u32,str
1,4,"""Personalized sentiment classif…"
2,5,"""Multi-source aggregated classi…"
3,4,"""TraceNet: Tracing and locating…"
4,7,"""SKEAFN: Sentiment Knowledge En…"
5,4,"""Enhancing aspect-category sent…"
6,6,"""Aspect-based sentiment analysi…"
7,4,"""Leveraging hierarchical langua…"
8,5,"""HOMOCHAR: A novel adversarial …"
9,4,"""Quantum Fuzzy Neural Network f…"


In [44]:
columnas_requeridas = [
    "id_articulo",
    "T1",  # Título
    "AU",  # Autor
    "PY",  # Año
    "KW",  # Palabras clave
    "DO",  # DOI
    "AB"   # Resumen
]

df_seleccionado = df_con_id.select(columnas_requeridas)

print("Dimensiones:", df_seleccionado.shape)
print("Columnas:", df_seleccionado.columns)

df_seleccionado.head(15)

Dimensiones: (1529, 7)
Columnas: ['id_articulo', 'T1', 'AU', 'PY', 'KW', 'DO', 'AB']


id_articulo,T1,AU,PY,KW,DO,AB
i64,str,str,str,str,str,str
1,"""Personalized sentiment classif…","""Zhang, You""","""2021""","""Sentiment analysis""","""https://doi.org/10.1016/j.knos…","""Incorporating extra attributes…"
1,null,"""Wang, Jin""",null,"""Text classification""",null,null
1,null,"""Zhang, Xuejie""",null,"""Sentence representation""",null,null
1,null,null,null,"""Bilinear attention""",null,null
2,"""Multi-source aggregated classi…","""Ma, Yu""","""2023""","""Stock prediction""","""https://doi.org/10.1016/j.inff…","""Predicting stock price movemen…"
…,…,…,…,…,…,…
3,null,"""Liu, Junfeng""",null,"""Sentiment analysis""",null,null
3,null,"""Kang, Zhongfeng""",null,"""Key items""",null,null
3,null,"""Zhou, Zenghui""",null,"""Interpretability""",null,null


In [45]:
autores_por_articulo = (
    df_seleccionado
    .group_by("id_articulo", maintain_order=True)
    .agg(
        pl.col("AU")
        .drop_nulls()
        .alias("AU")
    )
)

print("Dimensiones:", autores_por_articulo.shape)
print("Tipos:", autores_por_articulo.schema)

autores_por_articulo.head(10)

Dimensiones: (260, 2)
Tipos: Schema({'id_articulo': Int64, 'AU': List(String)})


id_articulo,AU
i64,list[str]
1,"[""Zhang, You"", ""Wang, Jin"", ""Zhang, Xuejie""]"
2,"[""Ma, Yu"", ""Mao, Rui"", … ""Cambria, Erik""]"
3,"[""Zhao, Qinghua"", ""Liu, Junfeng"", … ""Zhou, Zenghui""]"
4,"[""Zhu, Chuanbo"", ""Chen, Min"", … ""Chen, Jincai""]"
5,"[""Liu, Bin"", ""Lin, Tao"", ""Li, Ming""]"
6,"[""Zhao, Guoshuai"", ""Luo, Yiling"", … ""Qian, Xueming""]"
7,"[""Lengkeek, Matteo"", ""van der Knaap, Finn"", ""Frasincar, Flavius""]"
8,"[""Bajaj, Ashish"", ""Vishwakarma, Dinesh Kumar""]"
9,"[""Tiwari, Prayag"", ""Zhang, Lailei"", … ""Muhammad, Ghulam""]"


In [46]:
autores_por_articulo.filter(
    pl.col("id_articulo") == 1
)

id_articulo,AU
i64,list[str]
1,"[""Zhang, You"", ""Wang, Jin"", ""Zhang, Xuejie""]"


In [47]:
palabras_clave_por_articulo = (
    df_seleccionado
    .group_by("id_articulo", maintain_order=True)
    .agg(
        pl.col("KW")
        .drop_nulls()
        .alias("KW")
    )
)

print("Dimensiones:", palabras_clave_por_articulo.shape)
print("Tipos:", palabras_clave_por_articulo.schema)

palabras_clave_por_articulo.head(10)

Dimensiones: (260, 2)
Tipos: Schema({'id_articulo': Int64, 'KW': List(String)})


id_articulo,KW
i64,list[str]
1,"[""Sentiment analysis"", ""Text classification"", … ""Bilinear attention""]"
2,"[""Stock prediction"", ""Event-driven investing"", … ""Sentiment analysis""]"
3,"[""Language models"", ""Sentiment analysis"", … ""Interpretability""]"
4,"[""Multi-view learning"", ""Multiple feature fusion"", … ""Multi-head attention""]"
5,"[""Aspect-category sentiment analysis"", ""Data augmentation"", … ""Pretrained language model""]"
6,"[""Aspect-based sentiment analysis"", ""Aspect polarity classification"", … ""Multi-head attention""]"
7,"[""Text data"", ""Financial aspect classes"", … ""Hierarchical structure of data""]"
8,"[""Adversarial attack"", ""Vulnerability"", … ""Sentiment classification""]"
9,"[""Multimodal fusion"", ""Quantum Neural Networks"", … ""Fuzzy logic""]"


In [48]:
palabras_clave_por_articulo.filter(
    pl.col("id_articulo") == 1
)

id_articulo,KW
i64,list[str]
1,"[""Sentiment analysis"", ""Text classification"", … ""Bilinear attention""]"


In [49]:
palabras_clave_por_articulo.select(
    pl.col("KW")
    .list.eval(pl.element().is_null())
    .list.any()
    .sum()
    .alias("listas_con_nulos")
)

listas_con_nulos
u32
0


In [50]:
fragmentos_ab = (
    df_seleccionado
    .group_by("id_articulo", maintain_order=True)
    .agg(
        pl.col("AB").count().alias("cantidad_fragmentos")
    )
)

fragmentos_ab.filter(
    pl.col("cantidad_fragmentos") > 1
)

id_articulo,cantidad_fragmentos
i64,u32
27,3
96,3
206,10
253,8


In [51]:
resumenes_por_articulo = (
    df_seleccionado
    .group_by("id_articulo", maintain_order=True)
    .agg(
        pl.col("AB")
        .str.join(" ", ignore_nulls=True)
        .alias("AB")
    )
)

print("Dimensiones:", resumenes_por_articulo.shape)
print("Tipos:", resumenes_por_articulo.schema)

Dimensiones: (260, 2)
Tipos: Schema({'id_articulo': Int64, 'AB': String})


In [52]:
resumenes_por_articulo.select(
    "id_articulo",
    pl.col("AB")
    .str.slice(0, 200)
    .alias("inicio_resumen")
).head(10)

id_articulo,inicio_resumen
i64,str
1,"""Incorporating extra attributes…"
2,"""Predicting stock price movemen…"
3,"""We study sentiment analysis ta…"
4,"""Multimodal sentiment analysis …"
5,"""The goal of aspect-category se…"
6,"""Aspect based sentiment analysi…"
7,"""Every day millions of news art…"
8,"""State-of-the-art deep learning…"
9,"""Sentiment and sarcasm detectio…"


In [53]:
df_final = (
    df_seleccionado
    .group_by("id_articulo", maintain_order=True)
    .agg(
        pl.col("T1")
        .drop_nulls()
        .first()
        .alias("T1"),

        pl.col("AU")
        .drop_nulls()
        .alias("AU"),

        pl.col("PY")
        .drop_nulls()
        .first()
        .cast(pl.Int64, strict=False)
        .alias("PY"),

        pl.col("KW")
        .drop_nulls()
        .alias("KW"),

        pl.col("DO")
        .drop_nulls()
        .first()
        .alias("DO"),

        pl.col("AB")
        .str.join(" ", ignore_nulls=True)
        .alias("AB")
    )
    .drop("id_articulo")
)

df_final.head(10)

T1,AU,PY,KW,DO,AB
str,list[str],i64,list[str],str,str
"""Personalized sentiment classif…","[""Zhang, You"", ""Wang, Jin"", ""Zhang, Xuejie""]",2021,"[""Sentiment analysis"", ""Text classification"", … ""Bilinear attention""]","""https://doi.org/10.1016/j.knos…","""Incorporating extra attributes…"
"""Multi-source aggregated classi…","[""Ma, Yu"", ""Mao, Rui"", … ""Cambria, Erik""]",2023,"[""Stock prediction"", ""Event-driven investing"", … ""Sentiment analysis""]","""https://doi.org/10.1016/j.inff…","""Predicting stock price movemen…"
"""TraceNet: Tracing and locating…","[""Zhao, Qinghua"", ""Liu, Junfeng"", … ""Zhou, Zenghui""]",2023,"[""Language models"", ""Sentiment analysis"", … ""Interpretability""]","""https://doi.org/10.1016/j.knos…","""We study sentiment analysis ta…"
"""SKEAFN: Sentiment Knowledge En…","[""Zhu, Chuanbo"", ""Chen, Min"", … ""Chen, Jincai""]",2023,"[""Multi-view learning"", ""Multiple feature fusion"", … ""Multi-head attention""]","""https://doi.org/10.1016/j.inff…","""Multimodal sentiment analysis …"
"""Enhancing aspect-category sent…","[""Liu, Bin"", ""Lin, Tao"", ""Li, Ming""]",2023,"[""Aspect-category sentiment analysis"", ""Data augmentation"", … ""Pretrained language model""]","""https://doi.org/10.1016/j.knos…","""The goal of aspect-category se…"
"""Aspect-based sentiment analysi…","[""Zhao, Guoshuai"", ""Luo, Yiling"", … ""Qian, Xueming""]",2023,"[""Aspect-based sentiment analysis"", ""Aspect polarity classification"", … ""Multi-head attention""]","""https://doi.org/10.1016/j.knos…","""Aspect based sentiment analysi…"
"""Leveraging hierarchical langua…","[""Lengkeek, Matteo"", ""van der Knaap, Finn"", ""Frasincar, Flavius""]",2023,"[""Text data"", ""Financial aspect classes"", … ""Hierarchical structure of data""]","""https://doi.org/10.1016/j.ipm.…","""Every day millions of news art…"
"""HOMOCHAR: A novel adversarial …","[""Bajaj, Ashish"", ""Vishwakarma, Dinesh Kumar""]",2023,"[""Adversarial attack"", ""Vulnerability"", … ""Sentiment classification""]","""https://doi.org/10.1016/j.enga…","""State-of-the-art deep learning…"
"""Quantum Fuzzy Neural Network f…","[""Tiwari, Prayag"", ""Zhang, Lailei"", … ""Muhammad, Ghulam""]",2023,"[""Multimodal fusion"", ""Quantum Neural Networks"", … ""Fuzzy logic""]","""https://doi.org/10.1016/j.inff…","""Sentiment and sarcasm detectio…"


In [54]:
print("Dimensiones:", df_final.shape)
print("Columnas:", df_final.columns)
print("Tipos:", df_final.schema)

Dimensiones: (260, 6)
Columnas: ['T1', 'AU', 'PY', 'KW', 'DO', 'AB']
Tipos: Schema({'T1': String, 'AU': List(String), 'PY': Int64, 'KW': List(String), 'DO': String, 'AB': String})


In [55]:
df_final.null_count()

T1,AU,PY,KW,DO,AB
u32,u32,u32,u32,u32,u32
0,0,0,0,0,0


In [56]:
assert df_final.shape == (260, 6)

assert df_final.columns == [
    "T1", "AU", "PY", "KW", "DO", "AB"
]

assert df_final.null_count().sum_horizontal().item() == 0

print("Tabla final validada correctamente")

Tabla final validada correctamente


In [57]:
from pathlib import Path

# Copia preparada exclusivamente para Excel
df_excel = df_final.with_columns(
    pl.concat_str(
        [pl.lit("["), pl.col("AU").list.join("; "), pl.lit("]")],
        separator=""
    ).alias("AU"),

    pl.concat_str(
        [pl.lit("["), pl.col("KW").list.join("; "), pl.lit("]")],
        separator=""
    ).alias("KW")
)

SALIDA = ARCHIVO.parent / "lab01_resultado.xlsx"

df_excel.write_excel(
    workbook=SALIDA,
    worksheet="Resultado",
    table_style="Table Style Medium 2",
    freeze_panes=(1, 0),
    column_widths={
        "T1": 300,
        "AU": 260,
        "PY": 70,
        "KW": 260,
        "DO": 260,
        "AB": 500
    }
)

print(f"Archivo guardado en: {SALIDA}")

Archivo guardado en: C:\GitHub\tec-data-analisis\lab01\task\lab01_resultado.xlsx
